In [1]:
import functools
import xarray as xr
import pyearthtools.pipeline
class TemporalWindow(pyearthtools.pipeline.controller.PipelineIndex):
    '''
    The purpose of this class is to provide the ability to perform
    sequence-to-sequence modelling from an data accessor or pipeline
    that was designed to produce single time steps (i.e. single samples).

    The temporal window allows the specification of the 'back window'
    and the 'forward window', and will produce a binary branch.

    For example, if the time steps are hourly, and the base pipeline
    can produce hours 1, 2, 3 ... 10; then this Temporal Window can
    be used to produce sequence pairs like:
         [1,2,3], [4], 
         [2,3,4], [5],
         ...
         [7,8,9], [10]

    or like:
         [1,2], [3,4,5],
         [2,3], [4,5,6],
         ...
         [6,7], [8,9,10]

    This provides a simpler interface than the TemporalRetrieval which
    is a more general alternative.

    The window offsets are calculated not using positional indexing, but 
    using calculated date-times based on the reference time and the specified
    timedelta to calculate each required index exactly. The handling of missing
    data is left to the underlying pipeline response to the retrieval of the
    calculated datetime.

    The resultant sequences may be left unmerged (i.e. a list of retrieved
    results for each timetime) or merged (e.g. into an xarray along the time
    dimension). The default behaviour is to merge along the time dimension.

    A custom merge method may be specified.
    '''

    def __init__(self,
                 *,
                 a_indexes,
                 b_indexes,
                 timedelta,
                 merge_method=None
                ):
        '''
        Args:
            a_indexes: Multiplied by the timedelta then applied to the reference date
            b_indexes: Multiplied by the timedelta then applied to the reference date
            timedelta: Typically the time step of the underlying data
            merge_method: How to merge samples into a combined object
        '''
        self.a_indexes = a_indexes
        self.b_indexes = b_indexes
        self.timedelta = timedelta
        self.merge_method = merge_method

    def __getitem__(self, date_of_interest):

        date_of_interest = pyearthtools.data.Petdt(date_of_interest)

        head_i = [i * self.timedelta for i in self.a_indexes]
        tail_i = [i * self.timedelta for i in self.b_indexes]

        head = [self.parent_pipeline()[str(date_of_interest + delta)] for delta in head_i]
        tail = [self.parent_pipeline()[str(date_of_interest + delta)] for delta in tail_i]

        if self.merge_method:
            head = self.merge_method(head)
            tail = self.merge_method(tail)

        return head, tail
        


In [2]:
import os

# Most users should change this to the current directory.
os.environ['PETPROJECT'] = os.path.expanduser("~") + '/dev/proj/petcache'
workdir = os.environ['PETPROJECT']
# print(workdir)

import pathlib
import xarray as xr
from pathlib import Path
import time

import pyearthtools.data.archive
import pyearthtools.tutorial
import pyearthtools.pipeline


In [3]:
file_location = workdir + '/mini.nc'

In [4]:
accessor = pyearthtools.tutorial.ERA5DataClass.ERA5LowResDemoIndex([
                 '10m_u_component_of_wind', 
                 '10m_v_component_of_wind', 
                 'mean_sea_level_pressure',
                 '2m_temperature'    
],
filename_override=file_location)

In [5]:
sample = accessor['2010-01-01T00']

In [6]:
timedelta = pyearthtools.data.TimeDelta((1, "day"))

merge_method = functools.partial(xr.concat, dim='time')

In [7]:
data_pipeline = pyearthtools.pipeline.Pipeline(
    accessor,
    pyearthtools.data.transforms.coordinates.StandardLongitude(type="-180-180"),     
    # pyearthtools.pipeline.modifications.TemporalRetrieval(
    #     concat=True, samples=((0, 1), (6, 1, 6)) # Input = 1 sample from time T=0 hours. Output = T+6,+12,+18,+24
    # ),     
    TemporalWindow(a_indexes=[-3,-2,-1], b_indexes=[0], timedelta=timedelta, merge_method=merge_method),
    sampler=pyearthtools.pipeline.samplers.Default(),
    iterator=pyearthtools.pipeline.iterators.DateRange(1980, 2016, interval='6 hours')
)

In [8]:
doi = '20100101T0000'
sample = data_pipeline[doi]

/Users/munin/dev/proj/PyEarthTools/packages/data/src/pyearthtools/data/indexes/_indexes.py:809: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(


In [9]:
len(sample)

2

In [10]:
sample[0]

<xarray.Dataset> Size: 99kB
Dimensions:                  (time: 3, longitude: 64, latitude: 32)
Coordinates:
  * latitude                 (latitude) float64 256B -87.19 -81.56 ... 87.19
  * time                     (time) datetime64[ns] 24B 2009-12-29 ... 2009-12-31
  * longitude                (longitude) float64 512B -180.0 -174.4 ... 174.4
Data variables:
    10m_u_component_of_wind  (time, longitude, latitude) float32 25kB 0.01296...
    10m_v_component_of_wind  (time, longitude, latitude) float32 25kB -0.1656...
    2m_temperature           (time, longitude, latitude) float32 25kB 251.8 ....
    mean_sea_level_pressure  (time, longitude, latitude) float32 25kB 9.964e+...

In [11]:
sample[1]

<xarray.Dataset> Size: 34kB
Dimensions:                  (time: 1, longitude: 64, latitude: 32)
Coordinates:
  * latitude                 (latitude) float64 256B -87.19 -81.56 ... 87.19
  * time                     (time) datetime64[ns] 8B 2010-01-01
  * longitude                (longitude) float64 512B -180.0 -174.4 ... 174.4
Data variables:
    10m_u_component_of_wind  (time, longitude, latitude) float32 8kB 0.9633 ....
    10m_v_component_of_wind  (time, longitude, latitude) float32 8kB 2.515 .....
    2m_temperature           (time, longitude, latitude) float32 8kB 249.6 .....
    mean_sea_level_pressure  (time, longitude, latitude) float32 8kB 1.007e+0...